---
## 0. Setup


In [0]:
# Environment variables
catalog = "main"
schema  = "school"
volume  = "raw_data"

base_path        = f"/Volumes/{catalog}/{schema}/{volume}"
enrollments_path = f"{base_path}/enrollments"
courses_path     = f"{base_path}/courses"
students_path    = f"{base_path}/students"

print("Base path:", base_path)
print("Enrollments path:", enrollments_path)
print("Courses path:", courses_path)
print("Students path:", students_path)


Base path: /Volumes/main/school/raw_data
Enrollments path: /Volumes/main/school/raw_data/enrollments
Courses path: /Volumes/main/school/raw_data/courses
Students path: /Volumes/main/school/raw_data/students


In [0]:
# Create structure if it does not exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
print("List structure")

List structure


In [0]:
# Copy data from S3 to Volumes
s3_base = "s3://dalhussein-books/DEA-Book/datasets/school/v1"

dbutils.fs.cp(f"{s3_base}/enrollments",   enrollments_path, recurse=True)
dbutils.fs.cp(f"{s3_base}/courses-csv",   courses_path,     recurse=True)
dbutils.fs.cp(f"{s3_base}/students-json", students_path,    recurse=True)
print("Data copied successfully")


Data copied successfully


In [0]:
# Create enrollments table (parquet format)
spark.sql(f""" 
    CREATE TABLE IF NOT EXISTS {catalog}.{schema}.enrollments 
    AS SELECT * FROM parquet.`{enrollments_path}`
""")

# Create courses table from CSV
courses_df = spark.read.csv(courses_path, header=True, inferSchema=True, sep=";")
courses_df.write.mode("ignore").saveAsTable(f"{catalog}.{schema}.courses")

# Create students table from JSON
students_df = spark.read.json(students_path)
students_df.write.mode("ignore").saveAsTable(f"{catalog}.{schema}.students")

print("Tables enrollments, courses and student lists")


Tables enrollments, courses and student lists


In [0]:
# Register the tables as default to use %sql without catalog.schema
spark.sql(f"USE {catalog}.{schema}")
print(f"Using: {catalog}.{schema}")

Using: main.school


---
## 1. `FILTER` Function (Extracting elements from an array based on a condition)

The `enrollments` table contains a `courses` column, which is an **array of structs**.

`FILTER` iterates through this array and keeps **only the elements that meet the condition** specified in the lambda function.

**General Syntax:**
```sql
FILTER(array_col, element -> boolean_condition)
```

In [0]:
%sql
-- Preview of enrollments table
-- Note that the 'courses' column is an array of structs
SELECT enroll_id, student_id, courses
FROM enrollments
LIMIT 5

enroll_id,student_id,courses
000000000003559,S00001,"List(List(C09, 70, 7.2))"
000000000004243,S00002,"List(List(C07, 15, 28.05), List(C06, 90, 2.2))"
000000000004321,S00003,"List(List(C04, 10, 18.0))"
000000000004392,S00004,"List(List(C08, 35, 26.65))"
000000000003673,S00005,"List(List(C01, 75, 12.25), List(C11, 80, 7.6))"


### 1.1 Apply `FILTER` for highly discounted courses

We want to create a `highly_discounted_courses` column that contains only courses whose `discount_percent` is **greater than or equal to 60%**.

In [0]:
%sql
-- FILTER applies a lambda function to each element of the 'courses' array
-- Only keeps courses with discount_percent >= 60
-- Note that empty arrays [] appear for enrollments without courses with that discount
SELECT
  enroll_id,
  courses,
  FILTER(courses, course -> course.discount_percent >= 60) AS highly_discounted_courses
FROM enrollments
LIMIT 10

enroll_id,courses,highly_discounted_courses
000000000003559,"List(List(C09, 70, 7.2))","List(List(C09, 70, 7.2))"
000000000004243,"List(List(C07, 15, 28.05), List(C06, 90, 2.2))","List(List(C06, 90, 2.2))"
000000000004321,"List(List(C04, 10, 18.0))",List()
000000000004392,"List(List(C08, 35, 26.65))",List()
000000000003673,"List(List(C01, 75, 12.25), List(C11, 80, 7.6))","List(List(C01, 75, 12.25), List(C11, 80, 7.6))"
000000000004464,"List(List(C01, 50, 24.5), List(C02, 30, 19.6))",List()
000000000003495,"List(List(C09, 60, 9.6), List(C02, 25, 21.0))","List(List(C09, 60, 9.6))"
000000000004105,"List(List(C08, 25, 30.75), List(C02, 20, 22.4))",List()
000000000003825,"List(List(C09, 70, 7.2))","List(List(C09, 70, 7.2))"
000000000004062,"List(List(C09, 50, 12.0))",List()


### 1.2 Remove empty arrays with subquery + `WHERE size(...) > 0`

The previous result has rows with empty arrays `[]`.

Since **we cannot directly reference a derived column in the `WHERE` clause**, we use a **subquery** to filter by the size of the resulting array.

In [0]:
%sql
-- Subquery to allow the use of WHERE clauses on the derived column 'highly_discounted_courses'
-- size() returns the number of elements in the array; > 0 removes empty elements
SELECT enroll_id, highly_discounted_courses
FROM (
  SELECT
    enroll_id,
    courses,
    FILTER(courses, course -> course.discount_percent >= 60) AS highly_discounted_courses
  FROM enrollments
)
WHERE size(highly_discounted_courses) > 0


enroll_id,highly_discounted_courses
000000000003559,"List(List(C09, 70, 7.2))"
000000000004243,"List(List(C06, 90, 2.2))"
000000000003673,"List(List(C01, 75, 12.25), List(C11, 80, 7.6))"
000000000003495,"List(List(C09, 60, 9.6))"
000000000003825,"List(List(C09, 70, 7.2))"
000000000003650,"List(List(C09, 95, 1.2), List(C12, 80, 6.0))"
000000000003951,"List(List(C09, 75, 6.0))"
000000000003525,"List(List(C10, 95, 2.2))"
000000000004370,"List(List(C08, 65, 14.35))"
000000000004202,"List(List(C07, 65, 11.55), List(C08, 95, 2.05))"


### 1.3 Variant (Filter by multiple conditions)

Lambda can also combine conditions with `AND` / `OR`.

In [0]:
%sql
-- Courses with discounts >= 40% and subtotal > 50
-- Demonstrates that lambda accepts compound conditions
SELECT enroll_id, highly_filtered_courses
FROM (
  SELECT
    enroll_id,
    FILTER(
      courses,
      course -> course.discount_percent >= 40 AND course.subtotal > 5.5
    ) AS highly_filtered_courses
  FROM enrollments
)
WHERE size(highly_filtered_courses) > 0
LIMIT 10


enroll_id,highly_filtered_courses
000000000003559,"List(List(C09, 70, 7.2))"
000000000003673,"List(List(C01, 75, 12.25), List(C11, 80, 7.6))"
000000000004464,"List(List(C01, 50, 24.5))"
000000000003495,"List(List(C09, 60, 9.6))"
000000000003825,"List(List(C09, 70, 7.2))"
000000000004062,"List(List(C09, 50, 12.0))"
000000000003650,"List(List(C12, 80, 6.0))"
000000000003951,"List(List(C09, 75, 6.0))"
000000000003525,"List(List(C07, 50, 16.5))"
000000000004370,"List(List(C08, 65, 14.35))"


---
## 2. `TRANSFORM` Function (Transform each element of an array)

`TRANSFORM` iterates through each element of the array and **applies a lambda expression**,

returning a new array with the transformed values.

**General Syntax:**
```sql
TRANSFORM(array_col, element -> transformation_expression)
```

### 2.1 `TRANSFORM` with scalar value (Calculate tax)

We apply a **20% tax** to the `subtotal` field of each course in the array.

In [0]:
%sql
-- TRANSFORM returns a new array with the transformed values
-- Here we calculate subtotal * 1.2 (20% tax) and round to 2 decimal places
-- The result is an array of numeric (scalar) values
SELECT
  enroll_id,
  courses,
  TRANSFORM(
    courses,
    course -> ROUND(course.subtotal * 1.2, 2)
  ) AS courses_after_tax
FROM enrollments
LIMIT 10


enroll_id,courses,courses_after_tax
000000000003559,"List(List(C09, 70, 7.2))",List(8.64)
000000000004243,"List(List(C07, 15, 28.05), List(C06, 90, 2.2))","List(33.66, 2.64)"
000000000004321,"List(List(C04, 10, 18.0))",List(21.6)
000000000004392,"List(List(C08, 35, 26.65))",List(31.98)
000000000003673,"List(List(C01, 75, 12.25), List(C11, 80, 7.6))","List(14.7, 9.12)"
000000000004464,"List(List(C01, 50, 24.5), List(C02, 30, 19.6))","List(29.4, 23.52)"
000000000003495,"List(List(C09, 60, 9.6), List(C02, 25, 21.0))","List(11.52, 25.2)"
000000000004105,"List(List(C08, 25, 30.75), List(C02, 20, 22.4))","List(36.9, 26.88)"
000000000003825,"List(List(C09, 70, 7.2))",List(8.64)
000000000004062,"List(List(C09, 50, 12.0))",List(14.4)


### 2.2 `TRANSFORM` Generating Structs (Preserving Multiple Fields)

By default, `TRANSFORM` extracts only the transformed value.

If we need to **preserve the `course_id` along with the calculated value**,
we can return a **struct** within the lambda.


In [0]:
%sql
-- We generate a struct with two fields inside TRANSFORM:
-- 1. course_id -> original course identifier
-- 2. subtotal_with_tax -> subtotal with 20% tax applied
-- The result is an array of structs, more informative than just numeric values
SELECT
  enroll_id,
  TRANSFORM(
    courses,
    course -> (
      course.course_id,
      ROUND(course.subtotal * 1.2, 2) AS subtotal_with_tax
    )
  ) AS courses_after_tax
FROM enrollments
LIMIT 10


enroll_id,courses_after_tax
000000000003559,"List(List(C09, 8.64))"
000000000004243,"List(List(C07, 33.66), List(C06, 2.64))"
000000000004321,"List(List(C04, 21.6))"
000000000004392,"List(List(C08, 31.98))"
000000000003673,"List(List(C01, 14.7), List(C11, 9.12))"
000000000004464,"List(List(C01, 29.4), List(C02, 23.52))"
000000000003495,"List(List(C09, 11.52), List(C02, 25.2))"
000000000004105,"List(List(C08, 36.9), List(C02, 26.88))"
000000000003825,"List(List(C09, 8.64))"
000000000004062,"List(List(C09, 14.4))"


### 2.3 Combining `FILTER` + `TRANSFORM`

We can **chain** both functions: first we filter the courses with high discounts, and then we transform each one to calculate its final price after taxes.

In [0]:
%sql
-- Step 1: FILTER -> keeps only courses with discount_percent >= 50
-- Step 2: TRANSFORM on the filtered array -> applies the 20% tax
-- Nesting demonstrates how to compose higher-order functions
SELECT
  enroll_id,
  TRANSFORM(
    FILTER(courses, course -> course.discount_percent >= 50),
    course -> (
      course.course_id,
      course.discount_percent,
      ROUND(course.subtotal * 1.2, 2) AS subtotal_with_tax
    )
  ) AS discounted_courses_with_tax
FROM enrollments
WHERE size(FILTER(courses, course -> course.discount_percent >= 50)) > 0
LIMIT 10


enroll_id,discounted_courses_with_tax
000000000003559,"List(List(C09, 70, 8.64))"
000000000004243,"List(List(C06, 90, 2.64))"
000000000003673,"List(List(C01, 75, 14.7), List(C11, 80, 9.12))"
000000000004464,"List(List(C01, 50, 29.4))"
000000000003495,"List(List(C09, 60, 11.52))"
000000000003825,"List(List(C09, 70, 8.64))"
000000000004062,"List(List(C09, 50, 14.4))"
000000000003650,"List(List(C09, 95, 1.44), List(C12, 80, 7.2))"
000000000003951,"List(List(C09, 75, 7.2))"
000000000003525,"List(List(C07, 50, 19.8), List(C10, 95, 2.64))"


---
## 3. SQL UDFs (Reusable Functions with Custom Logic)

**SQL UDFs (User-Defined Functions)** allow you to encapsulate custom logic  
inside a named function that can be reused across any SQL query.

Unlike Python/Scala/Java UDFs, **SQL UDFs are directly optimized by Spark SQL**,  
which improves performance on large datasets.  
Additionally, they are **persistent database objects**, meaning they survive across sessions.

**General syntax:**
```SQL
CREATE OR REPLACE FUNCTION function_name(parameter DATA_TYPE)
RETURNS RETURN_TYPE
RETURN sql_expression
```


### 3.1 Create a UDF (Convert GPA to Percentage)
We will create a UDF called `gpa_to_percentage` that converts a GPA (scale 0–4.0) to its equivalent percentage (scale 0–100).
The conversion assumes a scale of 4.0, multiplying by 25.


In [0]:
%sql
-- We create the UDF gpa_to_percentage
-- Parameter: gpa of type DOUBLE
-- Return: INT (percentage rounded to the nearest integer)
-- Logic: gpa * 25 converts the scale 4.0 to 100%
CREATE OR REPLACE FUNCTION gpa_to_percentage(gpa DOUBLE)
RETURNS INT
RETURN CAST(ROUND(gpa * 25) AS INT)


### 3.2 Applying the UDF in a query

Once created, the UDF is used **exactly like a native SQL function**.


In [0]:
%sql
-- We apply the gpa to percentage UDF to the gpa column of the students table.
-- The percentage_score column shows the result of the conversion.
SELECT
  student_id,
  gpa,
  gpa_to_percentage(gpa) AS percentage_score
FROM students
LIMIT 10


student_id,gpa,percentage_score
S00001,1.48,37
S00002,3.02,76
S00003,3.31,83
S00004,1.89,47
S00005,3.55,89
S00006,2.9,73
S00007,2.96,74
S00008,1.2,30
S00009,1.96,49
S00010,1.39,35


### 3.3 Inspecting the UDF with `DESCRIBE FUNCTION`
We can obtain information about a UDF without having to search the source code.

In [0]:
%sql
-- DESCRIBE FUNCTION displays: database, parameters, and return type
DESCRIBE FUNCTION gpa_to_percentage


function_desc
Function: main.school.gpa_to_percentage
Type: SCALAR
Input: gpa DOUBLE
Returns: INT


In [0]:
%sql
-- DESCRIBE FUNCTION EXTENDED also displays the function body (SQL logic).
DESCRIBE FUNCTION EXTENDED gpa_to_percentage

function_desc
Function: main.school.gpa_to_percentage
Type: SCALAR
Input: gpa DOUBLE
Returns: INT
Collation: UTF8_BINARY
Deterministic: true
Data Access: CONTAINS SQL
Configs: spark.connect.session.connectML.enabled=true
spark.connect.session.connectML.mlCache.memoryControl.maxModelSize=268435456
spark.connect.session.planCompression.threshold=10485760


---
## 4. UDFs with Complex Logic (`CASE WHEN` Inside a UDF)

SQL UDFs can include **any valid SQL expression**,  
including `CASE WHEN` statements to handle multiple conditions.

**Grading scale:**

| GPA (4.0 scale) | Letter Grade |
|---|---|
| 3.50 – 4.0 | A |
| 2.75 – 3.49 | B |
| 2.0 – 2.74 | C |
| Below 2.0 | F |


### 4.1 Create UDF with `CASE WHEN` (Grade Letter)


In [0]:
%sql
-- UDF with CASE WHEN to map GPA to a letter grade
-- Parameter: gpa DOUBLE
-- Return: STRING with the corresponding letter (A, B, C, F)
CREATE OR REPLACE FUNCTION get_letter_grade(gpa DOUBLE)
RETURNS STRING
RETURN CASE
  WHEN gpa >= 3.5              THEN 'A'
  WHEN gpa >= 2.75 AND gpa < 3.5  THEN 'B'
  WHEN gpa >= 2.0  AND gpa < 2.75 THEN 'C'
  ELSE 'F'
END


In [0]:
%sql
-- We apply both UDFs together in the same query
-- Each row displays: id, original gpa, equivalent percentage, and letter
SELECT
  student_id,
  gpa,
  gpa_to_percentage(gpa)  AS percentage_score,
  get_letter_grade(gpa)   AS letter_grade
FROM students
LIMIT 10


student_id,gpa,percentage_score,letter_grade
S00001,1.48,37,F
S00002,3.02,76,B
S00003,3.31,83,B
S00004,1.89,47,F
S00005,3.55,89,A
S00006,2.9,73,B
S00007,2.96,74,B
S00008,1.2,30,F
S00009,1.96,49,F
S00010,1.39,35,F


### 4.2 Using UDFs in More Complex Expressions

UDFs can also be used in `WHERE`, `GROUP BY`, `ORDER BY` clauses, or combined with other SQL functions.


In [0]:
%sql
-- We counted how many students correspond to each grade letter.
-- We demonstrated that the UDF can be used directly in GROUP BY.
SELECT
  get_letter_grade(gpa) AS letter_grade,
  COUNT(*)              AS total_students,
  ROUND(AVG(gpa), 2)    AS avg_gpa
FROM students
GROUP BY get_letter_grade(gpa)
ORDER BY letter_grade


letter_grade,total_students,avg_gpa
A,288,3.74
B,414,3.12
C,431,2.37
F,567,1.52


In [0]:
%sql
-- We filter using WHERE directly from the UDF
-- Only students with grades 'A' or 'B'
SELECT
  student_id,
  gpa,
  get_letter_grade(gpa) AS letter_grade
FROM students
WHERE get_letter_grade(gpa) IN ('A', 'B')
ORDER BY gpa DESC
LIMIT 10


student_id,gpa,letter_grade
S01045,4.0,A
S01571,4.0,A
S01569,3.99,A
S01568,3.99,A
S01070,3.99,A
S01644,3.99,A
S00515,3.99,A
S01030,3.99,A
S00546,3.98,A
S00209,3.98,A


---
## 5. Managing UDFs (Inspecting and Deleting)

UDFs are persistent objects in the database.
We can list, inspect, and delete them when they are no longer needed.

In [0]:
%sql
-- List all available functions in the current schema
-- Includes native Spark functions and user-created UDFs
SHOW USER FUNCTIONS


function
main.school.get_letter_grade
main.school.gpa_to_percentage


In [0]:
%sql
-- We reviewed the extended definition of get_letter_grade again.
-- The 'Body' field shows the CASE WHEN we defined.
DESCRIBE FUNCTION EXTENDED get_letter_grade


function_desc
Function: main.school.get_letter_grade
Type: SCALAR
Input: gpa DOUBLE
Returns: STRING COLLATE UTF8_BINARY
Collation: UTF8_BINARY
Deterministic: true
Data Access: CONTAINS SQL
Configs: spark.connect.session.connectML.enabled=true
spark.connect.session.connectML.mlCache.memoryControl.maxModelSize=268435456
spark.connect.session.planCompression.threshold=10485760


### 5.1 Removing UDFs with `DROP FUNCTION`

We use `DROP FUNCTION` to remove a UDF from the database.

After executing these statements, the functions will **no longer be available**.


In [0]:
%sql
-- We removed both UDFs from the schema.
-- If we try to use them after this point, Spark will throw an error.
DROP FUNCTION IF EXISTS gpa_to_percentage;
DROP FUNCTION IF EXISTS get_letter_grade;


In [0]:
%sql
-- We verified that they no longer appear in the user functions list
SHOW USER FUNCTIONS


function


---
## Clean Up


In [0]:
def clean_up():
    print("Removing UDFs (if they exist)...")
    spark.sql("DROP FUNCTION IF EXISTS gpa_to_percentage")
    spark.sql("DROP FUNCTION IF EXISTS get_letter_grade")

    print("Removing tables...")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.enrollments")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.courses")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.students")

    print("Removing files from the volume...")
    dbutils.fs.rm(enrollments_path, True)
    dbutils.fs.rm(courses_path,     True)
    dbutils.fs.rm(students_path,    True)

    print("Removing schema...")
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")

    print("Done")

In [0]:
#clean_up()